# 08 — Spring Core

**What you'll learn:**

- The problem Spring solves: **inversion of control** and **dependency injection**
- What an `ApplicationContext` is and why every Spring app has one
- **Beans** — Spring-managed objects, registered by name and type
- **Component scanning** — `@Component`, `@Service`, `@Repository`, `@Controller`
- **`@Configuration`** classes and explicit `@Bean` methods
- Wiring dependencies with `@Autowired` — and why constructor injection wins
- **Bean scopes** — singleton, prototype, request, session
- The **bean lifecycle** — `@PostConstruct`, `@PreDestroy`
- **`@Profile`** — wiring different beans per environment
- Externalised configuration — `@Value`, `Environment`, `@ConfigurationProperties`
- **Aspect-Oriented Programming** — what aspects, pointcuts, and advice are
- `@Aspect` with `@Before`, `@After`, `@Around`
- `ApplicationContext` events — `@EventListener` and `ApplicationEventPublisher`

This notebook is where the Java foundation pays off. Every Spring feature you'll meet here uses the annotation-plus-reflection mechanics from notebook 07. By the end you'll understand the engine that the rest of the Spring track sits on.

## The problem: manual wiring

Imagine a small service. A `PaymentController` needs a `PaymentService`. The `PaymentService` needs a `PaymentRepository` and a `FraudCheck`. The `FraudCheck` needs an `HttpClient` and a `RuleEngine`. Without help, you write something like this:

```java
var http      = new HttpClient(config.timeout(), config.retries());
var rules     = new RuleEngine(rulesPath);
var fraud     = new FraudCheck(http, rules);
var repo      = new PaymentRepository(dataSource);
var service   = new PaymentService(repo, fraud);
var controller = new PaymentController(service);
```

Multiply that by every component in a real service and the wiring becomes the largest, ugliest, most fragile file in the project. Every time you add a constructor parameter, every caller has to change. Every component knows how to build its dependencies — concrete classes, concrete config — so swapping for a test or a different environment requires rewriting the constructor calls.

**Spring takes over that wiring.** You describe each component once, declare what it needs, and Spring builds the graph for you.

## Inversion of Control and Dependency Injection

The pattern has two names that get used interchangeably.

- **Inversion of Control (IoC)** — your code no longer controls *when* and *how* its dependencies get created. The framework does.
- **Dependency Injection (DI)** — the specific technique by which the framework supplies a component's dependencies, usually through its constructor.

```text
   without DI:                       with DI:
   class A {                         class A {
     B b = new B(...);                 A(B b) { this.b = b; }
     ...                               ...
   }                                 }
   A owns the decision to create B   A says "I need a B"
                                     Spring decides which B to inject
```

The benefit isn't shorter code — it's *swappability*. You can wire a fake `B` for tests, a different `B` for staging, the real `B` in production, all without touching `A`.

## The `ApplicationContext`

Every Spring app has an **`ApplicationContext`** — the container that holds every Spring-managed object (every **bean**), looks them up by type or name, manages their lifecycle, and broadcasts events to them.

```text
   ApplicationContext
   +---------------------------------------------------------+
   |                                                         |
   |   "paymentController" -> PaymentController instance     |
   |   "paymentService"    -> PaymentService instance        |
   |   "paymentRepository" -> PaymentRepository instance     |
   |   "fraudCheck"        -> FraudCheck instance            |
   |   "httpClient"        -> HttpClient instance            |
   |                                                         |
   +---------------------------------------------------------+
```

You don't construct beans yourself; you tell Spring *which classes are beans* and *what each one depends on*. Spring resolves the graph, instantiates the beans in dependency order, and injects what each needs.

## Component scanning — `@Component` and its specialisations

The simplest way to register a bean is to put `@Component` on the class. When Spring's component scanner walks the classpath, it picks up every class with `@Component` (or any annotation meta-annotated with `@Component`) and registers an instance as a bean.

Four annotations that all *are* `@Component`, but signal different roles:

| Annotation | Role |
|---|---|
| `@Component` | Generic Spring-managed bean |
| `@Service` | Business-logic / domain service |
| `@Repository` | Data-access object — also enables JDBC exception translation |
| `@Controller` / `@RestController` | Web layer — handles HTTP requests |

The differences in behaviour are small (`@Repository` adds some exception translation, `@Controller` adds web routing). The differences in *intent* matter — they tell the reader what layer a class belongs to.

In [ ]:
import org.springframework.stereotype.*;

@Repository
public class PaymentRepository {
    public Payment findById(long id) { /* talks to the database */ }
}

@Service
public class PaymentService {

    private final PaymentRepository repository;
    private final FraudCheck fraudCheck;

    // CONSTRUCTOR injection — Spring sees one constructor and wires it
    public PaymentService(PaymentRepository repository, FraudCheck fraudCheck) {
        this.repository = repository;
        this.fraudCheck = fraudCheck;
    }

    public Payment process(long id) {
        var payment = repository.findById(id);
        fraudCheck.evaluate(payment);
        return payment;
    }
}

@Component
public class FraudCheck {
    public void evaluate(Payment p) { /* ... */ }
}

Two important details. First, Spring uses constructor injection automatically when the class has a single constructor — no `@Autowired` annotation needed. Second, the bean *name* defaults to the class name with a lower-cased first letter (`paymentService`, `paymentRepository`), but you'd almost never refer to that name directly — you wire by **type**.

## `@Configuration` and `@Bean` — explicit registration

`@Component` scanning is great for your own classes, but you can't put `@Component` on a class you don't own — like an SDK class from a library. For those, write a `@Configuration` class with `@Bean` methods.

In [ ]:
import org.springframework.context.annotation.*;

@Configuration
public class HttpConfig {

    // each @Bean method returns an object that Spring registers as a bean
    @Bean
    public HttpClient httpClient(Environment env) {
        return HttpClient.newBuilder()
            .connectTimeout(Duration.ofSeconds(env.getRequiredProperty("http.timeout", Integer.class)))
            .build();
    }

    @Bean
    public RuleEngine ruleEngine() {
        return new RuleEngine(Path.of("rules.yml"));
    }
}

A `@Configuration` class is itself a `@Component`, so Spring picks it up during scanning. Each `@Bean` method returns the bean instance; the method's *parameters* become dependencies that Spring will resolve and pass in. This is the explicit form of dependency injection — the method body is your factory code.

## Three injection styles — and which to use

Spring supports three places to inject dependencies. Two of them are mistakes.

In [ ]:
// 1. CONSTRUCTOR INJECTION — preferred
@Service
public class GoodService {
    private final PaymentRepository repository;

    public GoodService(PaymentRepository repository) {
        this.repository = repository;
    }
}

// 2. SETTER INJECTION — avoid
@Service
public class OkayService {
    private PaymentRepository repository;

    @Autowired
    public void setRepository(PaymentRepository repository) {
        this.repository = repository;
    }
}

// 3. FIELD INJECTION — actively bad, but still common
@Service
public class BadService {
    @Autowired
    private PaymentRepository repository;     // bypasses every test isolation strategy
}

**Constructor injection wins** on every dimension. The field can be `final` (immutable). The class can be instantiated in plain unit tests without Spring. The compiler refuses to let you forget a dependency. The class screams when it has too many dependencies, which is a useful design signal. Use it everywhere; reach for setter injection only when a dependency is genuinely optional, and never reach for field injection.

When a class has just one constructor — which is the norm with the constructor-injection style — Spring autowires it implicitly. The `@Autowired` annotation is only needed when you have multiple constructors and want Spring to pick one.

## Bean scopes

Every bean has a **scope** — how many instances exist and how long they live. The two you'll use most:

- **`singleton`** (default) — one shared instance for the entire `ApplicationContext`. Created eagerly at startup.
- **`prototype`** — a fresh instance every time someone asks for it.

Web applications add two more: **`request`** (one instance per HTTP request) and **`session`** (one per HTTP session).

In [ ]:
import org.springframework.context.annotation.Scope;

@Component
public class SharedCache { }                  // singleton — the default

@Component
@Scope("prototype")
public class TempBuffer { }                   // new instance every time

@Component
@Scope(value = "request", proxyMode = ScopedProxyMode.TARGET_CLASS)
public class RequestContext { }               // one per HTTP request

**Singletons are the default for good reason** — your services are usually stateless, and one instance is enough. Reach for `prototype` only when each user of the bean genuinely needs its own copy. The trap to avoid: a singleton bean that injects a prototype is *still* a singleton; the prototype is resolved exactly once at construction. Use `ObjectProvider<T>` or `@Lookup` if you need a fresh prototype on every call.

## Bean lifecycle

After Spring constructs and injects a bean, it can give the bean a chance to finish initialising — and it can warn the bean before shutdown. The two annotations to know are `@PostConstruct` and `@PreDestroy`.

In [ ]:
import jakarta.annotation.*;

@Component
public class Cache {

    private Map<String, String> store;

    @PostConstruct
    public void init() {
        // runs AFTER constructor and AFTER all @Autowired dependencies are set
        this.store = new ConcurrentHashMap<>();
        warmUp();
    }

    @PreDestroy
    public void shutdown() {
        // runs when the ApplicationContext is closing
        store.clear();
    }
}

**The lifecycle order is:** construct → inject dependencies → `@PostConstruct` → bean is usable → application runs → context shutdown → `@PreDestroy`. Use `@PostConstruct` for any setup that depends on the injected fields being populated — you can't do that in the constructor itself, because the dependencies aren't there yet during a setter or field injection. (With constructor injection, they're already there, so `@PostConstruct` is rarer.)

## Profiles — different beans per environment

A real application needs different wiring in different environments. A real database in production; an in-memory one in tests; a stub for local dev. `@Profile` lets you tag beans with the environment they belong to.

In [ ]:
@Configuration
public class DataConfig {

    @Bean
    @Profile("prod")
    public DataSource productionDb() {
        return new HikariDataSource(/* real config */);
    }

    @Bean
    @Profile("!prod")          // active in everything EXCEPT prod
    public DataSource embeddedDb() {
        return new EmbeddedDatabaseBuilder().setType(H2).build();
    }
}

@Service
@Profile("dev | test")          // active in dev OR test
public class StubPaymentGateway implements PaymentGateway { /* ... */ }

You set the active profile by setting the `spring.profiles.active` property — in `application.properties`, as an environment variable, or via a `-D` flag. Only beans whose `@Profile` matches the active set get registered.

## Externalised configuration

Hard-coding values into Java is a smell — same code should run in dev, staging, and prod with different values. Spring reads configuration from many sources (properties files, YAML, environment variables, command-line flags) and exposes it three ways.

In [ ]:
// 1. @Value — inject a single property, with optional default
@Component
public class HttpClientWrapper {
    public HttpClientWrapper(
        @Value("${http.timeout:30}") int timeoutSeconds,
        @Value("${http.endpoint}") String endpoint
    ) { /* ... */ }
}

// 2. Environment — programmatic lookup
@Component
public class FeatureFlags {
    private final boolean experimentalEnabled;

    public FeatureFlags(Environment env) {
        this.experimentalEnabled = env.getProperty("feature.experimental", Boolean.class, false);
    }
}

// 3. @ConfigurationProperties — bind a whole group of keys to a typed record (best)
@ConfigurationProperties(prefix = "http")
public record HttpProps(String endpoint, int timeout, int retries) {}

@Service
public class HttpService {
    public HttpService(HttpProps props) { /* use props.endpoint(), props.timeout() */ }
}

Of the three, **`@ConfigurationProperties` is the right default**. It groups related properties under a prefix, binds them to a typed record (or class), validates at startup, and gives you autocomplete-friendly IDE support. Use `@Value` only for one-off cases.

## Aspect-Oriented Programming

There's a class of behaviour that doesn't belong to any one component but crosses many of them: logging, transactions, security checks, metrics, retries. These are **cross-cutting concerns**. AOP lets you write them once and apply them to many methods.

The vocabulary:

- **Aspect** — a class containing cross-cutting behaviour
- **Advice** — a method on the aspect that runs at some point (`@Before`, `@After`, `@Around`)
- **Pointcut** — an expression that picks which target methods the advice applies to
- **Join point** — a specific method execution that the advice intercepts

In [ ]:
import org.aspectj.lang.annotation.*;
import org.aspectj.lang.ProceedingJoinPoint;

@Aspect
@Component
public class TimingAspect {

    private static final Logger log = LoggerFactory.getLogger(TimingAspect.class);

    // pointcut: "any public method in any class in com.example.service"
    @Around("execution(public * com.example.service..*(..))")
    public Object time(ProceedingJoinPoint pjp) throws Throwable {
        long start = System.nanoTime();
        try {
            return pjp.proceed();           // run the real method
        } finally {
            long ms = (System.nanoTime() - start) / 1_000_000;
            log.info("{} took {} ms", pjp.getSignature().toShortString(), ms);
        }
    }

    // shorter forms — before / after, no proceed
    @Before("execution(* com.example.repo..*(..))")
    public void logRepoCall(JoinPoint jp) {
        log.debug("calling {}", jp.getSignature().toShortString());
    }
}

How it works: Spring creates a **proxy** around every bean whose methods match an aspect's pointcut. The proxy intercepts calls and runs the advice. This is the same proxying mechanism behind `@Transactional`, `@Cacheable`, and Spring Security's method security. AOP is not just a feature — it is a core mechanism several Spring features rely on.

**One gotcha**: AOP works on calls *into* a bean from outside. A method calling another method on the same bean (via `this.other()`) bypasses the proxy and skips the advice. If you need self-invocation to go through the aspect, inject the bean into itself or restructure.

## `ApplicationContext` events

Beans can broadcast events to each other through the container. This decouples a producer from the (possibly many) consumers of an event — the producer doesn't know who listens.

In [ ]:
import org.springframework.context.*;
import org.springframework.context.event.EventListener;

// the event — any class is allowed; a record is ideal
public record PaymentProcessed(long paymentId, long amount, Instant at) {}

// the publisher — get the ApplicationEventPublisher injected
@Service
public class PaymentService {
    private final ApplicationEventPublisher events;

    public PaymentService(ApplicationEventPublisher events) {
        this.events = events;
    }

    public void process(Payment p) {
        // ... do work ...
        events.publishEvent(new PaymentProcessed(p.id(), p.amount(), Instant.now()));
    }
}

// any number of listeners — Spring discovers them via reflection (sound familiar?)
@Component
public class AuditListener {
    @EventListener
    public void onPayment(PaymentProcessed event) {
        // log to audit table
    }
}

@Component
public class MetricsListener {
    @EventListener
    public void onPayment(PaymentProcessed event) {
        // increment a counter
    }
}

Events are synchronous by default — `publishEvent` blocks until every listener returns. Add `@Async` to a listener method (and `@EnableAsync` to a config class) for asynchronous delivery. Spring also ships built-in events like `ContextRefreshedEvent` and `ContextClosedEvent` you can listen for to run startup/shutdown hooks.

## Putting it together

A small but realistic Spring application, end to end:

In [ ]:
// the configuration root
@SpringBootApplication                       // = @Configuration + @ComponentScan + @EnableAutoConfiguration
public class App {
    public static void main(String[] args) {
        SpringApplication.run(App.class, args);
    }
}

// configuration properties bound from application.yml
@ConfigurationProperties(prefix = "payment")
public record PaymentProps(int maxRetries, Duration timeout) {}

// repository — data layer
@Repository
public class PaymentRepository {
    public Optional<Payment> findById(long id) { /* JDBC or JPA */ }
}

// service — domain layer, constructor-injected
@Service
public class PaymentService {
    private final PaymentRepository repository;
    private final ApplicationEventPublisher events;
    private final PaymentProps props;

    public PaymentService(PaymentRepository repository,
                          ApplicationEventPublisher events,
                          PaymentProps props) {
        this.repository = repository;
        this.events = events;
        this.props = props;
    }

    public Payment process(long id) {
        var payment = repository.findById(id).orElseThrow();
        // ... real work, respecting props.maxRetries() and props.timeout() ...
        events.publishEvent(new PaymentProcessed(id, payment.amount(), Instant.now()));
        return payment;
    }
}

// cross-cutting: log every service method's duration
@Aspect @Component
public class TimingAspect {
    @Around("execution(* com.example.service..*(..))")
    public Object time(ProceedingJoinPoint pjp) throws Throwable {
        long start = System.nanoTime();
        try { return pjp.proceed(); }
        finally { log.info("{} took {} ns", pjp.getSignature(), System.nanoTime() - start); }
    }
}

Every annotation in that file is something Spring will read at startup via the same reflection mechanics from notebook 07. There is no XML, no factory configuration, no manual wiring. You declared what the components are and what they need; Spring constructed the graph.

## What's next

You can now describe a system to Spring and let it do the wiring. The pattern of *annotate the class, declare what it needs in the constructor, let the container do the rest* is the bedrock of every Spring notebook ahead.

Notebook 09 turns Spring Core into a real web service: **Spring Web**. We'll build REST controllers with `@RestController` and `@RequestMapping`, validate incoming requests with `@Valid`, handle errors centrally with `@ControllerAdvice`, call other services with `RestClient`/`WebClient`, and lock things down with the essentials of Spring Security.